# Notebook to evaluate initial code

In [ ]:
from pfns.scripts.tabpfn_interface import TabPFNClassifier
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from pfns.scripts.tabular_metrics import auc_metric
import numpy as np
import time


datasets = {
    "iris": load_iris,
    "wine": load_wine,
    "breast_cancer": load_breast_cancer
}

MAX_FEATURES = 25

base_path = "/home/david/ICL-Architectures/PFNs/models_diff/large_config_v3.pt/tabpfn_prior_config_very_large_0_no_seed/"

for name, loader in datasets.items():
    X, y = loader(return_X_y=True)
    X = X[:, :MAX_FEATURES]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # TabPFN
    start_time = time.time()
    clf_tabpfn = TabPFNClassifier(
        base_path=base_path,
        device="cuda:0",
        model_string="checkpoint.pt",
        N_ensemble_configurations=1000,
    )
    clf_tabpfn.fit(X_train, y_train)
    preds_proba_tabpfn = clf_tabpfn.predict_proba(X_test)
    time_tabpfn = time.time() - start_time
    
    # Get predictions from probabilities
    preds_tabpfn = np.argmax(preds_proba_tabpfn, axis=1)
    acc_tabpfn = accuracy_score(y_test, preds_tabpfn)
    roc_auc_tabpfn = auc_metric(y_test, preds_proba_tabpfn)
    
    # Random Forest
    start_time = time.time()
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    preds_proba_rf = rf.predict_proba(X_test)
    time_rf = time.time() - start_time
    
    preds_rf = np.argmax(preds_proba_rf, axis=1)
    acc_rf = accuracy_score(y_test, preds_rf)
    roc_auc_rf = auc_metric(y_test, preds_proba_rf)

    print(f"{name:>15} | TabPFN: Acc={acc_tabpfn:.4f} ROC={roc_auc_tabpfn:.4f} Time={time_tabpfn:.2f}s | RF: Acc={acc_rf:.4f} ROC={roc_auc_rf:.4f} Time={time_rf:.2f}s")